In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType,DoubleType
from pyspark.sql.functions import from_json, col, unix_timestamp

# Kafka Configuration
KAFKA_BROKER = "master:9092"  # Update with your Kafka broker address
USER_CLICK_TOPIC = "user_click"
USER_CLICK_TABLE = "user_click_events"

In [ ]:
# Initialize Spark Session with Hive Support
scala_version = '2.12'
spark_version = '3.1.2'
# TODO: Ensure match above values match the correct versions
packages = [
    f'org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}',
    'org.apache.kafka:kafka-clients:3.2.2'
]

spark = SparkSession.builder \
        .master("local") \
        .appName("Process User Click Events") \
        .config("spark.jars.packages", ",".join(packages))\
        .config("spark.sql.catalogImplementation", "hive") \
        .enableHiveSupport() \
        .getOrCreate()

In [ ]:
# Define Schema for User Click Events
user_click_schema = StructType([
    StructField("event_type", StringType(), True),
    StructField("user_id", LongType(), True),
    StructField("page", StringType(), True),
    StructField("device", StringType(), True),
    StructField("location", StringType(), True),
    StructField("timestamp", DoubleType(), True)
])

In [ ]:
# Step 1: Read Data from Kafka Topic
print("Reading data from Kafka topic...")
raw_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", USER_CLICK_TOPIC) \
    .load()

In [ ]:
# Step 2: Deserialize JSON and Apply Schema
print("Deserializing JSON data...")
user_click_df = raw_stream_df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), user_click_schema).alias("data")) \
    .select(
        col("data.event_type"),
        col("data.user_id"),
        col("data.page"),
        col("data.device"),
        col("data.location"),
        (col("data.timestamp").cast("long") / 1000).cast(TimestampType()).alias("event_time")
    )

# from pyspark.sql.functions import col, from_unixtime
# from_unixtime(col("data.timestamp")).cast(TimestampType()).alias("event_time")

In [ ]:
query = user_click_df.writeStream.outputMode("append") \
    .format("console") \
    .start()

query.awaitTermination()

In [ ]:
# Step 3: Write Data to Parquet Files with Trigger
print("Writing streaming data to Parquet files...")
checkpoint_dir = "/user/userjuly2025019/CaseStudy/checkpoint"
output_path = "/user/userjuly2025019/CaseStudy/UserClick/out/"

query = user_click_df.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_dir) \
    .trigger(processingTime="60 seconds") \
    .start()

query.awaitTermination()

In [ ]:
spark.sql('''CREATE TABLE if not exists user_clicks 
    ( 
        event_type STRING, 
        user_id BIGINT, 
        page STRING, 
        device STRING, 
        location STRING, 
        event_time TIMESTAMP 
    ) STORED AS PARQUET LOCATION '/user/userjuly2025019/CaseStudy/UserClick/out/';
''')

spark.sql('''select * from user_clicks''').show()

In [ ]:
df = spark.read.parquet("/user/userjuly2025019/CaseStudy/UserClick/out/*.parquet")

# Show data
df.show()

# Print schema
df.printSchema()

In [ ]:
df.createOrReplaceTempView("parquet_table")
spark.sql("select * from parquet_table").show()

In [ ]:
spark.sql("SHOW TABLES").show()